## this notebook is used for user-written SQL query-derived cohort building

adapted from notebook:

"HM-hypothyroidism-id-v1.ipynb"

in workspace "Hypothyroidism genomics v7"


In [ ]:
# !pip install polars
# !pip install matplotlib-venn

In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import json
import re
from google.cloud import bigquery
import polars as pl
import gcsfs
from matplotlib_venn import venn2
from tqdm.notebook import tqdm

In [ ]:
from datetime import datetime
start = datetime.now()
start

In [ ]:
def wb(*args):
    """Run a wb command and return parsed JSON."""
    cmd = ["wb", *args, "--format=json"]
    result = subprocess.check_output(cmd, text=True)
    return json.loads(result)


# Get workspace info
workspace = wb("workspace", "describe")
GOOGLE_CLOUD_PROJECT = workspace["googleProjectId"]

# Get resources
resources = wb("resource", "list")

# WORKSPACE_BUCKET
bucket_resources = [
    r for r in resources
    if r.get("resourceType") == "GCS_BUCKET"
    and "practical_considerations_bucket" in r.get("id", "")
    and "temporary" not in r.get("id", "")
]

if not bucket_resources:
    raise ValueError("No matching bucket found")

WORKSPACE_BUCKET = f"gs://{bucket_resources[0]['bucketName']}"

# WORKSPACE_CDR
bq_resources = [
    r for r in resources
    if r.get("resourceType") in {"BQ_DATASET", "BIGQUERY_DATASET"}
]

cdr_resources = [
    r for r in bq_resources
    if re.match(r"^C\d{4}Q\d+R\d+$", r.get("datasetId", ""))
]

if not cdr_resources:
    raise ValueError("No matching CDR dataset found")

WORKSPACE_CDR = (
    f"{cdr_resources[0]['projectId']}."
    f"{cdr_resources[0]['datasetId']}"
)

In [ ]:
#get variables
version = WORKSPACE_CDR
bucket = WORKSPACE_BUCKET
cohort = "allofus"
google_project_id = GOOGLE_CLOUD_PROJECT

### helper funx

In [ ]:
def polars_gbq(query):
    """
    Take a SQL query and return result as polars dataframe
    :param query: BigQuery SQL query
    :return: polars dataframe
    """
    client = bigquery.Client()
    query_job = client.query(query)
    rows = query_job.result()
    df = pl.from_arrow(rows.to_arrow())

    return df


In [ ]:
def view_query(sql_query, nrows = 5):
    df = pd.read_gbq(f"""
        SELECT * FROM ({sql_query})
        LIMIT {nrows}""", dialect="standard")
    return df

def view_tb(sql_query, nrows = 5):
    df = pd.read_gbq(f"""
        SELECT * FROM {sql_query}
        LIMIT {nrows}""", dialect="standard")
    return df

def count_query(sql_query):
    df = pd.read_gbq(f"""
        SELECT COUNT(DISTINCT person_id) person_cnt
        FROM ({sql_query})
        """, dialect="standard")
    return df

def count_grp_query(sql_query, groupby_col, min_count=None):
    df = pd.read_gbq(f"""
        SELECT {groupby_col}, COUNT(DISTINCT person_id) person_cnt
        FROM ({sql_query})
        GROUP BY {groupby_col}
        ORDER BY person_cnt DESC
        """, dialect="standard")
    if min_count is not None:
        return df[df['person_cnt'] > min_count]
    return df

def make_like_str(ls, colname): 
    return " OR ".join(f'{colname} LIKE "{x}%"\n' for x in ls)

def make_icd_str(icd9_in_ls, icd9_like_ls, icd10_in_ls, icd10_like_ls, version = version):
    icd_where_ls = []
    if(len(icd9_in_ls) > 0): 
        icd_where_ls.append('c1.concept_code in (\n' + ', '.join(['"' + x + '"' for x in icd9_in_ls]) + '\n)' + \
        " AND c1.vocabulary_id = 'ICD9CM'")
    if(len(icd10_in_ls) > 0): 
        icd_where_ls.append('c1.concept_code in (\n' + ', '.join(['"' + x + '"' for x in icd10_in_ls]) + '\n)' + \
        " AND c1.vocabulary_id = 'ICD10CM'")
    if(len(icd9_like_ls) > 0): 
        icd_where_ls.append('(\n' + make_like_str(icd9_like_ls, 'c1.concept_code') + '\n)' + \
        " AND c1.vocabulary_id = 'ICD9CM'")
    if(len(icd10_like_ls) > 0):
        icd_where_ls.append('(\n' + make_like_str(icd10_like_ls, 'c1.concept_code') + '\n)' + \
        " AND c1.vocabulary_id = 'ICD10CM'")
    icd_where_str = "OR\n".join(["(\n" + x + "\n)\n" for x in icd_where_ls])
    re_sql = f"""
        SELECT DISTINCT
            cond.person_id, cond.condition_concept_id, cond.condition_start_date, 
            cstd.concept_name, 
            c1.concept_code, c1.concept_name icd_name, c1.vocabulary_id
        FROM
            {version}.concept c1 
            INNER JOIN
            {version}.concept_relationship crel
                ON crel.concept_id_1 = c1.concept_id
            INNER JOIN
            {version}.concept cstd
                ON crel.concept_id_2 = cstd.concept_id
            INNER JOIN
            {version}.condition_occurrence cond 
                ON cond.condition_concept_id = cstd.concept_id
        WHERE
            cstd.standard_concept = "S" AND
            crel.relationship_id = "Maps to" AND
            (
            {icd_where_str}
            )
        """
    return re_sql

In [ ]:
# This one intend to add support to concept hierarchy with concept_ancestor table

def make_icd_str2(icd9_in_ls, icd9_like_ls, icd10_in_ls, icd10_like_ls, version = version):
    icd_where_ls = []
    if(len(icd9_in_ls) > 0): 
        icd_where_ls.append('c1.concept_code in (\n' + ', '.join(['"' + x + '"' for x in icd9_in_ls]) + '\n)' + \
        " AND c1.vocabulary_id = 'ICD9CM'")
    if(len(icd10_in_ls) > 0): 
        icd_where_ls.append('c1.concept_code in (\n' + ', '.join(['"' + x + '"' for x in icd10_in_ls]) + '\n)' + \
        " AND c1.vocabulary_id = 'ICD10CM'")
    if(len(icd9_like_ls) > 0): 
        icd_where_ls.append('(\n' + make_like_str(icd9_like_ls, 'c1.concept_code') + '\n)' + \
        " AND c1.vocabulary_id = 'ICD9CM'")
    if(len(icd10_like_ls) > 0):
        icd_where_ls.append('(\n' + make_like_str(icd10_like_ls, 'c1.concept_code') + '\n)' + \
        " AND c1.vocabulary_id = 'ICD10CM'")
    icd_where_str = "OR\n".join(["(\n" + x + "\n)\n" for x in icd_where_ls])
    re_sql = f"""
        SELECT DISTINCT
            cond.person_id, cond.condition_concept_id, cond.condition_start_date, 
            cstd.concept_name, 
            c1.concept_code, c1.concept_name icd_name, c1.vocabulary_id
        FROM
            {version}.concept c1 
            INNER JOIN
            {version}.concept_relationship crel
                ON crel.concept_id_1 = c1.concept_id
            INNER JOIN
            {version}.concept cstd
                ON crel.concept_id_2 = cstd.concept_id
            LEFT JOIN
            {version}.concept_ancestor anc on cstd.concept_id = anc.ancestor_concept_id
            INNER JOIN
            {version}.condition_occurrence cond 
                ON cond.condition_concept_id = cstd.concept_id 
                    OR cond.condition_concept_id = anc.descendant_concept_id
        WHERE
            cstd.standard_concept = "S" AND
            crel.relationship_id = "Maps to" AND
            (
            {icd_where_str}
            )
        """
    return re_sql

In [ ]:
def make_ingr2concept_query(ingr_list, version = version): 
    where_string = " OR ".join("LOWER(c.concept_name) LIKE '%" 
                               + str(x.lower()) + "%'" for x in ingr_list)
    drug_concept_query = """
        SELECT DISTINCT 
            c2.concept_name,
            c2.concept_code,
            c2.concept_id,
            c2.vocabulary_id,
            c.concept_id ingr_concept_id,
            c.concept_name ingr_name,
            c.concept_code ingr_code
        FROM
            `{0}.concept` c
        INNER JOIN `{0}.concept_ancestor` ca
            ON c.concept_id = ca.ancestor_concept_id
        INNER JOIN `{0}.concept` c2
            ON c2.concept_id = ca.descendant_concept_id
        WHERE
            c.concept_class_id = 'Ingredient'
            AND ({1})
        """.format(version, where_string)
    return(drug_concept_query)

In [ ]:
def make_drug_exposure_query(ingr_list, version = version): 
    concept_query = make_ingr2concept_query(ingr_list, version)
    exposure_query = """
        SELECT DISTINCT
            d.person_id, 
            d.drug_concept_id, 
            d.drug_exposure_start_date,
            d.drug_type_concept_id,
            d.route_concept_id,
            d.visit_occurrence_id, 
            dc.ingr_concept_id,
            dc.ingr_name
        FROM
            `{0}.drug_exposure` d INNER JOIN 
            ({1}) dc ON d.drug_concept_id = dc.concept_id
    """.format(version, concept_query)
    return (exposure_query)

In [ ]:
def make_drug_exposure_stat_query(ingr_list, version = version): 
    concept_query = make_ingr2concept_query(ingr_list, version)
    stat_query = """
        SELECT
            dc.ingr_concept_id,
            dc.ingr_name, 
            COUNT(DISTINCT d.person_id) person_cnt
        FROM
            `{0}.drug_exposure` d INNER JOIN 
            ({1}) dc ON d.drug_concept_id = dc.concept_id
        GROUP BY 
            dc.ingr_concept_id,
            dc.ingr_name
        ORDER BY person_cnt DESC
    """.format(version, concept_query)
    return(stat_query)

In [ ]:
def make_measurement_query(loinc_str, criteria_str, version = version): 
    re_query = f"""
        SELECT
            m.person_id,
            m.value_as_number,
            m.measurement_date,
            m.measurement_concept_id,
            m.unit_source_value, 
            m.visit_occurrence_id,
            c.concept_name,
            c.concept_code
        FROM
            {version}.measurement m INNER JOIN
            {version}.concept c ON m.measurement_concept_id = c.concept_id
        WHERE
            c.vocabulary_id='LOINC' AND c.concept_code = '{loinc_str}'
            AND m.value_as_number {criteria_str}
        """
    return(re_query)

In [ ]:
def make_measurement_query2(loinc, criteria_str, version = version): 
    re_query = f"""
        SELECT
            m.person_id,
            m.value_as_number,
            m.measurement_date,
            m.measurement_concept_id,
            m.unit_source_value, 
            m.visit_occurrence_id,
            c.concept_name,
            c.concept_code
        FROM
            {version}.measurement m INNER JOIN
            {version}.concept c ON m.measurement_concept_id = c.concept_id
        WHERE
            m.measurement_concept_id IN {tuple(loinc)}
            AND m.value_as_number {criteria_str}
        """
    return(re_query)

## Case Group

### Case Inclusion
Case inclusion criteria: (all three conditions required): </br>
*	ICD9 code for hypothyroidism OR abnormal TSH/FT4 
*	Thyroid replacement medication use
*	Require at least 2 instances of either medication or lab (a combination is acceptable) with at least 3 months between the first and last instance of medication or lab

#### Case ICD codes </br>
 
ICD-9-CM:  </br>
244 acquired hypothyroidism </br>
244.8  acquired hypothyroidism NEC </br>
244.9  hypothyroidism NOS </br>
245  thyroiditis </br>
245.2  chronic lymphocytic thyroiditis </br>
245.8 chronic thyroiditis NEC/NOS </br>
245.9  thyroiditis NOS </br>

In [ ]:
# Update if necessary

hypoTh_icd9_ls = ['244', 
                  '244.8',  
                  '244.9',  
                  '245', 
                  '245.2', 
                  '245.8', 
                  '245.9']
#hypoTh_icd9_str = ', '.join(['"' + x + '"' for x in hypoTh_icd9_ls])
#hypoTh_icd9_str


In [ ]:
hypoTh_icd10_ls = ['E06',
                  'E06.2',
                  'E06.5',
                  'E03.8',
                    'E03.9',
                  'E06.9']   


#hypoTh_icd10_str = ', '.join(['"' + x + '"' for x in hypoTh_icd10_ls])
#hypoTh_icd10_str


In [ ]:
# Look at concept_relationship table for ICD9 inclusion codes
icd9_code_tuple_str = tuple(hypoTh_icd9_ls)

icd9_mapping = pd.read_gbq(f'''
SELECT 
    condition_occurrence.condition_source_value,
    concept_relationship.concept_id_1,
    concept_relationship.concept_id_2,
    concept_relationship.relationship_id,
    concept.concept_name,
    concept.vocabulary_id AS relationship_vocab,
    concept.concept_class_id
FROM
    {version}.condition_occurrence condition_occurrence
    INNER JOIN {version}.concept_relationship concept_relationship
    ON condition_occurrence.condition_source_concept_id = concept_relationship.concept_id_1
    INNER JOIN {version}.concept
    ON concept_relationship.concept_id_2 = concept.concept_id
WHERE
    condition_occurrence.condition_source_value IN {icd9_code_tuple_str}
    AND 
    concept.standard_concept = 'S'
''')

In [ ]:
pd.read_gbq(f'''
SELECT DISTINCT
    vocabulary_id
FROM
    {version}.concept
WHERE
    vocabulary_id LIKE 'ICD%'

''')

In [ ]:
pd.read_gbq(f'''
SELECT DISTINCT
    condition_occurrence.condition_concept_id,
    condition_occurrence.condition_source_value,
    concept2.vocabulary_id,
    concept.concept_name
FROM
    {version}.condition_occurrence condition_occurrence
    INNER JOIN {version}.concept concept
    ON condition_occurrence.condition_concept_id = concept.concept_id
    INNER JOIN {version}.concept concept2
    ON condition_occurrence.condition_source_concept_id = concept2.concept_id
WHERE
    (condition_occurrence.condition_source_value IN {tuple(hypoTh_icd9_ls)})
    OR
    (condition_occurrence.condition_source_value IN {tuple(hypoTh_icd10_ls)})

''')

In [ ]:
icd9_mapping.filter(items = ['concept_id_2', 'concept_name', 'relationship_vocab']).drop_duplicates()

In [ ]:
# This might have duplicate due to mapping (No hierarchy)
hypoTh_icd_query0_old = make_icd_str(icd9_in_ls = hypoTh_icd9_ls, 
             icd9_like_ls = [], 
             icd10_in_ls = hypoTh_icd10_ls, icd10_like_ls = [])

In [ ]:
# This might have duplicate due to mapping (with hierarchy)
hypoTh_icd_query0 = make_icd_str2(icd9_in_ls = hypoTh_icd9_ls, 
             icd9_like_ls = [], 
             icd10_in_ls = hypoTh_icd10_ls, icd10_like_ls = [])

In [ ]:
# Cleaned up query
hypoTh_icd_query1 = f"""
SELECT DISTINCT 
    person_id, 
    condition_concept_id, 
    condition_start_date, 
    concept_name
FROM ({hypoTh_icd_query0})
"""

In [ ]:
# Alternative cleaned up query
# updated on 20220928 for a typo. This query was not further used no no impact
hypoTh_icd_query2 = f"""
WITH 
cond AS (
SELECT 
    person_id, 
    COUNT(DISTINCT condition_start_date) cond_date_cnt,
    MIN(condition_start_date) first_cond_date, 
FROM 
    ({hypoTh_icd_query0})
GROUP BY person_id
)
SELECT
    cond.person_id, cond.cond_date_cnt, cond.first_cond_date, 
    DATE_DIFF(cond.first_cond_date, DATE(p.birth_datetime), YEAR) first_cond_age
FROM
    cond INNER JOIN
    {version}.person p USING (person_id)
"""

In [ ]:
# statistics
pd.read_gbq(f"""
SELECT
    concept_code, vocabulary_id, icd_name,
    COUNT(DISTINCT person_id) person_cnt
FROM ({hypoTh_icd_query0})
GROUP BY
    concept_code, vocabulary_id, icd_name
HAVING
    COUNT(DISTINCT person_id) >= 20
""", dialect="standard")

In [ ]:
pd.read_gbq(f"""
SELECT
    COUNT(DISTINCT person_id) person_cnt
FROM ({hypoTh_icd_query0})
""", dialect="standard")

#### Laboratories

In [ ]:
tsh_loinc = '3016-3'

tsh_inc = 'BETWEEN 5 AND 1000'

In [ ]:
tsh_inc_query = make_measurement_query(loinc_str = tsh_loinc, criteria_str = tsh_inc)

In [ ]:
tsh_units = pd.read_gbq(f'''
SELECT 
    COUNT(DISTINCT person_id) AS person_count,
    unit_source_value
FROM
    ({tsh_inc_query})
GROUP BY
    unit_source_value
ORDER BY 
    person_count DESC
''')

print(tsh_units[tsh_units['person_count'] > 20])

In [ ]:
tsh_units_list = [i for i in tsh_units.unit_source_value]
tsh_units_conv = []
for i in tsh_units_list:
    if str(i).isdigit() == True:
            tsh_units_conv.append(
                pd.read_gbq(f'''
                    SELECT DISTINCT
                        concept_name
                    FROM
                        {version}.concept
                    WHERE
                        concept_id = {int(i)}
                ''').concept_name
            )
    elif str(i).isdigit() == False:
        tsh_units_conv.append(str(i)) 

In [ ]:
pd.DataFrame({'ID': tsh_units_list, 'converted': tsh_units_conv})

In [ ]:
pd.read_gbq(f'''
                    SELECT DISTINCT
                        concept_name
                    FROM
                        {version}.concept
                    WHERE
                        concept_id = 3314211000001106
                ''')

In [ ]:
count_query(tsh_inc_query)

In [ ]:
# Thyroxine (T4) free [Mass/volume] in Serum or Plasma

fT4_loinc = '3024-7' 

# Need to verify units
# A typical normal range is 0.9 to 2.3 nanograms per deciliter (ng/dL), 
# or 12 to 30 picomoles per liter (pmol/L).
# Per PheKB: Hypothyroidism: TSH >5 or FT4 <0.5

fT4_dec = '< 0.5'

In [ ]:
fT4_dec_query = make_measurement_query(loinc_str = fT4_loinc, criteria_str = fT4_dec)

In [ ]:
count_query(fT4_dec_query)

In [ ]:
# Thyroglobulin Ab [Units/volume] in Serum or Plasma by Immunoassay
# https://www.mayocliniclabs.com/test-catalog/overview/84382#Clinical-and-Interpretive

tg_ab_loinc = '56536-6' # '8098-6' ??? 
tg_ab_inc = '> 4.0' 

In [ ]:
tg_ab_inc_query = make_measurement_query(loinc_str = tg_ab_loinc, criteria_str = tg_ab_inc)

In [ ]:
count_query(tg_ab_inc_query)

In [ ]:
# Thyroperoxidase Ab [Units/volume] in Serum or Plasma
# https://www.mayocliniclabs.com/test-catalog/overview/81765#Clinical-and-Interpretive

tpo_ab_loinc = '8099-4'
tpo_ab_inc = '> 9.0'

In [ ]:
tpo_ab_inc_query = make_measurement_query(loinc_str = tpo_ab_loinc, criteria_str = tpo_ab_inc)

In [ ]:
count_query(tpo_ab_inc_query)

#### Medications

In [ ]:
# need to be completed

replacement_med_ls = ['levothyroxine', 
                      'liothyronine', 'liotrix', 
                      'thyroid desiccated', 
                      'desiccated thyroid', 
                      'triiodothyronine']  # Generic names only; upper vs lower cases NOT matters

In [ ]:
replacement_med_query = make_drug_exposure_query(ingr_list = replacement_med_ls)

In [ ]:
count_query(replacement_med_query)

In [ ]:
count_grp_query(replacement_med_query, 'ingr_name')

In [ ]:
pd.read_gbq(make_drug_exposure_stat_query(ingr_list = replacement_med_ls), dialect="standard")

#### Pregnancy

In [ ]:
preg_icd9_ls = ['V22.1', 'V22.2', '631', '633', '633.0', '633.00', '633.1', 
                '633.10', '633.20', '633.8', '633.80', '633.9', '633.90', 
                '645.1', '645.2', '646.8', '646.9', '648.1', '651', '651.0', 
                '651.1', '651.2', '651.8', '651.9', '651.90', '761.4', 'V23.89', 
                'V61.6', 'V61.7']
# Need to add ICD10? 

In [ ]:
preg_icd_query = make_icd_str(icd9_in_ls = preg_icd9_ls, 
             icd9_like_ls = [], 
             icd10_in_ls = [], icd10_like_ls = [])

In [ ]:
count_query(preg_icd_query)

In [ ]:
# HCG: https://www.labcorp.com/tests/004416/human-chorionic-gonadotropin-hcg-subunit-quantitative
# Choriogonadotropin.intact+Beta subunit [Units/volume] in Serum or Plasma

hcg_loinc = '45194-8'
hcg_inc = '> 5'

In [ ]:
hcg_inc_query = make_measurement_query(loinc_str = hcg_loinc, criteria_str = hcg_inc)

In [ ]:
count_query(hcg_inc_query)

In [ ]:
# Choriogonadotropin.intact+Beta subunit [Mass/volume] in Serum or Plasma

count_query(make_measurement_query('93769-8', '> 0'))

In [ ]:
# hCG, TOTAL MoM: 23841-0
# https://www.mayocliniclabs.com/test-catalog/overview/113145#Fees-and-Codes

count_query(make_measurement_query('23841-0', '> 0'))

In [ ]:
# hCG, TOTAL
# https://www.mayocliniclabs.com/test-catalog/overview/113145#Fees-and-Codes

count_query(make_measurement_query('83086-9', '> 0'))

In [ ]:
# https://www.labcorp.com/tests/004556/human-chorionic-gonadotropin-hcg-subunit-qualitative

count_query(make_measurement_query('2110-5', '> 6'))


In [ ]:
# Unite IU/mL (1/1000 of regular hCG)
# First Trimester Screen w/NT
# https://www.labcorp.com/tests/017500/first-trimester-screen-with-nuchal-translucency

preg_hcg_screen_query = make_measurement_query('19080-1', '> 0')

count_query(preg_hcg_screen_query)

In [ ]:
count_query(make_measurement_query('32166-1', '> 0'))

In [ ]:
urine_hcg_pos_query = f"""
        SELECT
            m.person_id,
            m.value_as_concept_id,
            c2.concept_name result_name,
            m.measurement_date,
            m.measurement_concept_id,
            m.visit_occurrence_id,
            c.concept_name,
            c.concept_code
        FROM
            {version}.measurement m INNER JOIN
            {version}.concept c ON m.measurement_concept_id = c.concept_id INNER JOIN
            {version}.concept c2 ON m.value_as_concept_id = c2.concept_id
        WHERE
            c.vocabulary_id='LOINC' AND c.concept_code = '2106-3'
            AND m.value_as_concept_id in (45884084, 45877985, 9191)
"""

In [ ]:
all_preg_query = f"""
SELECT DISTINCT
    person_id, 
    condition_start_date event_date, 
    'ICD' preg_flag
FROM ({preg_icd_query})
UNION ALL
SELECT DISTINCT
    person_id, 
    measurement_date event_date, 
    'plasma_hcg' preg_flag
FROM ({hcg_inc_query})
UNION ALL
SELECT DISTINCT
    person_id, 
    measurement_date event_date, 
    'hcg_trimester_screem' preg_flag
FROM ({preg_hcg_screen_query})
UNION ALL
SELECT DISTINCT
    person_id, 
    measurement_date event_date, 
    'hcg_urine' preg_flag
FROM ({urine_hcg_pos_query})
"""
count_grp_query(all_preg_query, 'preg_flag')

#### Criteria

In [ ]:
# inclusion criterion 1: 
# ICD9 code for hypothyroidism OR abnormal TSH/FT4
# Removed pregnancy-flagged tests

incl_sub_query_1_0 = f"""
SELECT DISTINCT
    person_id, condition_start_date date, 'icd' query
FROM ({hypoTh_icd_query0})
UNION ALL
SELECT DISTINCT
    m.person_id, m.measurement_date date, 'tsh' query
FROM 
    ({tsh_inc_query}) m LEFT JOIN
    ({all_preg_query}) preg ON m.person_id = preg.person_id 
        AND DATE_DIFF(m.measurement_date, preg.event_date, MONTH) BETWEEN -6 AND 12
WHERE
    preg.preg_flag IS NULL
UNION ALL
SELECT DISTINCT
    m.person_id, m.measurement_date date, 'fT4' query
FROM 
    ({fT4_dec_query}) m LEFT JOIN
    ({all_preg_query}) preg ON m.person_id = preg.person_id 
        AND DATE_DIFF(m.measurement_date, preg.event_date, MONTH) BETWEEN -6 AND 12
WHERE
    preg.preg_flag IS NULL
"""
count_query(incl_sub_query_1_0)

In [ ]:
# Without consider pregnancy

count_query(f"""
SELECT DISTINCT
    person_id, condition_start_date date, 'icd' query
FROM ({hypoTh_icd_query0})
UNION ALL
SELECT DISTINCT
    m.person_id, m.measurement_date date, 'tsh' query
FROM 
    ({tsh_inc_query}) m LEFT JOIN
    ({all_preg_query}) preg ON m.person_id = preg.person_id 
        AND DATE_DIFF(m.measurement_date, preg.event_date, MONTH) BETWEEN -6 AND 12
WHERE
    preg.preg_flag IS NULL
UNION ALL
SELECT DISTINCT
    m.person_id, m.measurement_date date, 'fT4' query
FROM 
    ({fT4_dec_query}) m
""")

In [ ]:
# Clean up criterion 1
incl_sub_query_1_1 = f"""
SELECT 
incl.person_id, 
MIN(incl.date) cr1_first_date,
MIN(DATE_DIFF(incl.date, DATE(p.birth_datetime), YEAR)) cr1_first_age
FROM 
    ({incl_sub_query_1_0}) incl INNER JOIN
    {version}.person p USING (person_id)
GROUP BY incl.person_id
"""

In [ ]:
# inclusion criterion 2:
# Thyroid replacement medication use

incl_sub_query_2 = f"""
SELECT 
incl.person_id, 
MIN(incl.drug_exposure_start_date) cr2_first_date,
MIN(DATE_DIFF(incl.drug_exposure_start_date, DATE(p.birth_datetime), YEAR)) cr2_first_age
FROM 
    ({replacement_med_query}) incl INNER JOIN
    {version}.person p USING (person_id)
GROUP BY incl.person_id
"""
count_query(incl_sub_query_2)

In [ ]:
# inclusion criterion 3:
# Require at least 2 instances of either medication or lab (a combination is acceptable) 
# with at least 3 months between the first and last instance of medication or lab

incl_sub_query_3_prep = f"""
SELECT
    person_id, measurement_date date, 'TSH' query
FROM ({tsh_inc_query})
UNION ALL
SELECT
    person_id, measurement_date date, 'fT4' query
FROM ({fT4_dec_query})
UNION ALL
SELECT
    person_id, measurement_date date, 'TG_Ab' query
FROM ({tg_ab_inc_query})
UNION ALL
SELECT
    person_id, measurement_date date, 'TPO_Ab' query
FROM ({tpo_ab_inc_query})
UNION ALL
SELECT
    person_id, drug_exposure_start_date date, 'Replacement_drug' query
FROM ({replacement_med_query})
"""

In [ ]:
# Clean up criterion 3
incl_sub_query_3 = f"""
SELECT 
incl.person_id, 
MIN(incl.date) cr3_first_date,
MAX(incl.date) cr3_last_date, 
COUNT(DISTINCT incl.date) cr3_occ_cnt
FROM 
    ({incl_sub_query_3_prep}) incl
GROUP BY incl.person_id
HAVING
    DATE_DIFF(cr3_last_date, cr3_first_date, DAY) >= 90 AND
    cr3_occ_cnt >= 2
"""
count_query(incl_sub_query_3)

In [ ]:
case_incl_query = f"""
SELECT
    cr1.person_id, 
    cr1.cr1_first_date,
    cr1.cr1_first_age, 
    cr2.cr2_first_date, 
    cr2.cr2_first_age,
    cr3.cr3_first_date, 
    cr3.cr3_last_date, 
    cr3.cr3_occ_cnt
FROM
    ({incl_sub_query_1_1}) cr1 INNER JOIN
    ({incl_sub_query_2}) cr2 USING (person_id) INNER JOIN
    ({incl_sub_query_3}) cr3 USING (person_id)
"""

count_query(case_incl_query)

In [ ]:
case_incl_ids = tuple(pd.read_gbq(case_incl_query)['person_id'])

### Case Exclusion

#### Secondary Hypothyroidism and thyroid diseases ICD

In [ ]:
# Update if necessary

excl_icd9_ls = ['242.0', '242.1', '242.2', '242.3', '242.9',
                '244.0', 
                '244.1', 
                '244.2', 
                '244.3']

excl_icd9_str = ', '.join(['"' + x + '"' for x in excl_icd9_ls])
excl_icd9_str


In [ ]:
excl_icd9_like_ls = ['193', '258']  # Need to make a function to deal with this: like "193%"

In [ ]:
# Please Add
# E89.0 Postprocedural hypothyroidism
# E01.8 Other iodine-deficiency related thyroid disorders and allied conditions
# E03.2 Hypothyroidism due to medicaments and other exogenous substances

# Removed ^above and included ICD10 codes mapped from phecodes

#['E89.0', 
#                  'E01.8', <- acquired hypothyroidism should not be here
#                 'E03.2']
excl_icd10_ls = ['E05.21',
 'E05.2',
 'E05.20',
 'C73',
 'Z85.850',
 'E05.9',
 'E05.90',
 'E05.10',
 'E05.80',
 'E05',
 'E05.4',
 'E05.31',
 'E05.91',
 'E05.40',
 'E05.11',
 'E05.81',
 'E05.41',
 'E05.3',
 'E05.30',
 'E05.1',
 'E03.2',
 'E89.0']
excl_icd10_str = ', '.join(['"' + x + '"' for x in excl_icd10_ls])
excl_icd10_str


ICD9 -> phecode -> ICD10 output: ICD10 codes below that were mapped using phecodes mapped to ICD9

In [ ]:
excl_icd10_ls_2 = ['E05.21',
 'E05.2',
 'E05.20',
 'C73',
 'Z85.850',
 'E05.9',
 'E05.90',
 'E05.10',
 'E05.80',
 'E05',
 'E05.4',
 'E05.31',
 'E05.91',
 'E05.40',
 'E05.11',
 'E05.81',
 'E05.41',
 'E05.3',
 'E05.30',
 'E05.1',
 'E03.2',
 'E89.0']
excl_icd10_str2 = ', '.join(['"' + x + '"' for x in excl_icd10_ls_2])
excl_icd10_str2
excl_icd10_ls_2_ = tuple(excl_icd10_ls_2)

In [ ]:
phecode_mapped_icd10_excl = view_query(f'''
SELECT
    COUNT(distinct person.person_id) AS person_cnt,
    condition_occurrence.condition_source_value,
    concept.concept_name,
    concept.vocabulary_id
FROM
    {version}.person person
    INNER JOIN {version}.condition_occurrence condition_occurrence
    ON person.person_id = condition_occurrence.person_id
    INNER JOIN {version}.concept concept
    ON condition_occurrence.condition_source_concept_id = concept.concept_id
WHERE
    condition_occurrence.condition_source_value IN {excl_icd10_ls_2_}
GROUP BY
    condition_occurrence.condition_source_value, concept.concept_name, concept.vocabulary_id
ORDER BY
    person_cnt DESC
''', nrows = len(excl_icd10_ls_2))


In [ ]:
sum(phecode_mapped_icd10_excl.person_cnt)

#### looking at the icd9 mappings to snomed, comparing them to cohort builder

Take the ICD9 codes from the emerge document, use concept relationship table to extract corresponding snomed codes

In [ ]:
# Look at ICD9 exclusion mappings
icd9_code_excl_tuple_str = tuple(excl_icd9_ls)

icd9_excl_mapping = pd.read_gbq(f'''
SELECT 
    condition_occurrence.condition_source_value,
    concept_relationship.concept_id_1,
    concept2.vocabulary_id AS og_vocab,
    concept_relationship.concept_id_2,
    concept_relationship.relationship_id,
    concept.concept_name,
    concept.vocabulary_id AS relationship_vocab,
    concept.concept_class_id
FROM
    {version}.condition_occurrence condition_occurrence
    INNER JOIN {version}.concept_relationship concept_relationship
    ON condition_occurrence.condition_source_concept_id = concept_relationship.concept_id_1
    INNER JOIN {version}.concept
    ON concept_relationship.concept_id_2 = concept.concept_id
    INNER JOIN {version}.concept concept2
    ON concept_relationship.concept_id_1 = concept2.concept_id
WHERE
    (condition_occurrence.condition_source_value IN {icd9_code_excl_tuple_str}
    OR condition_occurrence.condition_source_value LIKE '193%'
    OR condition_occurrence.condition_source_value LIKE '258%')
    AND 
    concept.standard_concept = 'S'
    AND
    concept2.vocabulary_id = 'ICD9CM'
ORDER BY
    condition_occurrence.condition_source_value
''')

icd9_excl_mapping.head()

In [ ]:
icd9_excl_mapped_to_snomed = icd9_excl_mapping.concept_id_2
icd9_excl_mapping.filter(items=['condition_source_value','concept_id_2','concept_name']).drop_duplicates()

In [ ]:
excl_icd_query = make_icd_str(icd9_in_ls = excl_icd9_ls, 
             icd9_like_ls = excl_icd9_like_ls, 
             icd10_in_ls = excl_icd10_ls, icd10_like_ls = [])
count_query(excl_icd_query)

In [ ]:
view_query(f'''
SELECT COUNT(DISTINCT person_id) person_cnt
FROM
({case_incl_query}) inc INNER JOIN
({excl_icd_query}) exc USING (person_id)
''')

In [ ]:
count_grp_query(excl_icd_query, 'concept_name', min_count=20)

#### •	Post surgical or post-radiation hypothyroidism (by ICD9 codes or CPT codes for the procedures)

In [ ]:
rad_tx_cpt = ['77261', '77262', '77263', '77280', '77285', '77290', 
              '77295', '77299', '77300', '77301', '77305', '77310', 
              '77315', '77321', '77326', '77327', '77328', '77331', 
              '77332', '77333', '77334', '77336', '77370', '77399', 
              '77401', '77402', '77403', '77404', '77406', '77408', 
              '77409', '77411', '77412', '77413', '77414', '77416', 
              '77417', '77418', '77427', '77431', '77432', '77470', 
              '77499', '77520', '77522', '77523', '77525', '77750', 
              '77761', '77762', '77763', '77776', '77777', '77778', 
              '77781', '77782', '77783', '77784', '77789', '77790', 
              '77799']

rad_tx_cpt_str = ', '.join(['"' + x + '"' for x in rad_tx_cpt])
rad_tx_cpt_str

In [ ]:
rad_tx_cpt_query = f"""
SELECT
    p.person_id, p.procedure_date, p.procedure_concept_id, p.visit_occurrence_id, 
    c.concept_code, c.concept_name
FROM
    {version}.procedure_occurrence p INNER JOIN
    {version}.concept c ON procedure_concept_id = c.concept_id
WHERE
    c.concept_code in ({rad_tx_cpt_str}) AND c.vocabulary_id = 'CPT4'
"""
count_query(rad_tx_cpt_query)

In [ ]:
count_grp_query(rad_tx_cpt_query, 'concept_name', min_count=20)

In [ ]:
thyroidectomy_cpt = ['60240', '60271', '60260', '60252', '60254', 
                     '60270', '60500', '60502', '60505', '60200', 
                     '78020', '60225', '60210', '60212', '60220']

thyroidectomy_cpt_str = ', '.join(['"' + x + '"' for x in thyroidectomy_cpt])
thyroidectomy_cpt_str

In [ ]:
thyroidectomy_cpt_query = f"""
SELECT
    p.person_id, p.procedure_date, p.procedure_concept_id, p.visit_occurrence_id, 
    c.concept_code, c.concept_name
FROM
    {version}.procedure_occurrence p INNER JOIN
    {version}.concept c ON procedure_concept_id = c.concept_id
WHERE
    c.concept_code in ({thyroidectomy_cpt_str}) AND c.vocabulary_id = 'CPT4'
"""

count_query(thyroidectomy_cpt_query)

In [ ]:
count_grp_query(thyroidectomy_cpt_query, 'concept_name').head()

#### •	Any thyroid-altering medication

In [ ]:
thy_alt_med_ls = ['Phenytoin', 'Amiodarone', 'Lithium', 'Methimazole', 'Propylthiouracil']

In [ ]:
thy_alt_med_query = make_drug_exposure_query(ingr_list = thy_alt_med_ls)

#### Exclusion Criteria Together

In [ ]:
case_excl_query = f"""
SELECT DISTINCT person_id, 'ICD' excl_criteria
FROM ({excl_icd_query})
UNION ALL
SELECT DISTINCT person_id, 'Radiation' excl_criteria
FROM ({rad_tx_cpt_query})
UNION ALL
SELECT DISTINCT person_id, 'Thyroidectomy' excl_criteria
FROM ({thyroidectomy_cpt_query})
UNION ALL
SELECT DISTINCT person_id, 'Thyroid-altering Medication' excl_criteria
FROM ({thy_alt_med_query})
"""
count_grp_query(case_excl_query, 'excl_criteria')

In [ ]:
count_query(case_excl_query)

In [ ]:
count_grp_query(f"""
SELECT
incl.person_id, excl.excl_criteria
FROM
({case_excl_query}) excl INNER JOIN
({case_incl_query}) incl USING (person_id)
""", 'excl_criteria')

In [ ]:
count_query(f"""
SELECT
incl.person_id, excl.excl_criteria
FROM
({case_excl_query}) excl INNER JOIN
({case_incl_query}) incl USING (person_id)
""")

In [ ]:
case_excl_query1 = f"""
SELECT person_id, ARRAY_AGG(DISTINCT excl_criteria) excl_criteria_grp
FROM ({case_excl_query})
GROUP BY person_id
"""

### Final Cases


In [ ]:
final_case_query = f"""
SELECT DISTINCT
    incl.*
FROM
    ({case_incl_query}) incl LEFT JOIN
    ({case_excl_query}) excl USING (person_id)  
WHERE
    excl.excl_criteria IS NULL
"""

count_query(final_case_query)

## Control Group

### Laboratories
* Must have a normal TSH (and FT4 if checked)
* Does it mean a normal TSH and exclude all who has abnormal TSH and FT4?

In [ ]:
tsh_norm_query = make_measurement_query(loinc_str = tsh_loinc, criteria_str = 'BETWEEN 0.5 AND 5') # need to verify

In [ ]:
count_query(tsh_norm_query)

In [ ]:
tsh_norm_query2 = make_measurement_query2(loinc = (3009201,3019170), criteria_str= 'BETWEEN 0.5 AND 5', version = version)
count_query(tsh_norm_query2)

In [ ]:
fT4_norm_query = make_measurement_query(loinc_str = fT4_loinc, criteria_str = 'BETWEEN 0.5 AND 1.2') 

In [ ]:
count_query(fT4_norm_query)

In [ ]:
ft4_norm_query2 = make_measurement_query2(loinc=(3008598,3016991), criteria_str = 'BETWEEN 0.5 AND 1.2', version= version)
count_query(ft4_norm_query2)

In [ ]:
# Please double check the WHERE clause for minimal condition count

norm_lab_ctrl_query = f"""
SELECT DISTINCT
    tsh.person_id, 
    tsh.TSH_cnt, 
    tsh.first_TSH_date, DATE_DIFF(tsh.first_TSH_date, DATE(p.birth_datetime), YEAR) first_TSH_age,
    tsh.last_TSH_date, DATE_DIFF(tsh.last_TSH_date, DATE(p.birth_datetime), YEAR) last_TSH_age,
    ft4.fT4_cnt, ft4.first_fT4_date, ft4.last_fT4_date,
    cond.cond_date_cnt
FROM
    (
        SELECT 
            person_id, 
            COUNT(DISTINCT measurement_date) TSH_cnt, 
            MIN(measurement_date) first_TSH_date,
            MAX(measurement_date) last_TSH_date
        FROM ({tsh_norm_query2})
        GROUP BY person_id
    ) tsh 
    LEFT JOIN
    (
        SELECT 
            person_id, 
            COUNT(DISTINCT measurement_date) fT4_cnt, 
            MIN(measurement_date) first_fT4_date,
            MAX(measurement_date) last_fT4_date

        FROM ({ft4_norm_query2})
        GROUP BY person_id
    ) ft4 USING (person_id)
    LEFT JOIN
    (
        SELECT person_id, COUNT(DISTINCT condition_start_date) cond_date_cnt
        FROM {version}.condition_occurrence
        GROUP BY person_id
    ) cond USING (person_id) INNER JOIN
    {version}.person p USING (person_id)
WHERE
    cond.cond_date_cnt >= 1 AND
    tsh.TSH_cnt >= 1
"""

count_query(norm_lab_ctrl_query)

### Control Exclusion

In [ ]:
# forgot to add control specific exclusions!
pd.read_gbq(f'''
SELECT
    COUNT(DISTINCT person_id) person_count
FROM
    {version}.condition_occurrence 
WHERE
    condition_source_value LIKE '240%' OR
    condition_source_value LIKE '241%' OR
    condition_source_value LIKE '242%' OR
    condition_source_value LIKE '243%' OR
    condition_source_value LIKE '244%' OR
    condition_source_value LIKE '245%' OR
    condition_source_value LIKE '246%' OR
    condition_source_value = '358.0' OR 
    condition_source_value = '358.00' OR
    condition_source_value = '358.01' OR
    condition_source_value = 'V58.0'
''')

In [ ]:
ctrl_excl_query = f"""
{case_excl_query}
UNION ALL
SELECT DISTINCT person_id, 'Hypothyroidism diagnosed' excl_criteria
FROM ({hypoTh_icd_query0})
UNION ALL
SELECT DISTINCT person_id, 'Increased TSH' excl_criteria
FROM ({tsh_inc_query})
UNION ALL
SELECT DISTINCT person_id, 'Decreased fT4' excl_criteria
FROM ({fT4_dec_query})
UNION ALL
SELECT DISTINCT person_id, 'Thyroid replacement medication' excl_criteria
FROM ({replacement_med_query})
UNION ALL
SELECT DISTINCT person_id, 'control exclusion' excl_criteria
FROM (
    SELECT
        DISTINCT person_id
    FROM
        {version}.condition_occurrence 
    WHERE
        condition_source_value LIKE '240%' OR
        condition_source_value LIKE '241%' OR
        condition_source_value LIKE '242%' OR
        condition_source_value LIKE '243%' OR
        condition_source_value LIKE '244%' OR
        condition_source_value LIKE '245%' OR
        condition_source_value LIKE '246%' OR
        condition_source_value = '358.0' OR 
        condition_source_value = '358.00' OR
        condition_source_value = '358.01' OR
        condition_source_value = 'V58.0'
)
"""

count_grp_query(ctrl_excl_query, 'excl_criteria')

### Final Control

In [ ]:
final_ctrl_query = f"""
SELECT DISTINCT
    incl.*
FROM
    ({norm_lab_ctrl_query}) incl LEFT JOIN
    ({ctrl_excl_query}) excl USING (person_id)
WHERE excl.excl_criteria IS NULL
"""

count_query(final_ctrl_query)

## Case and Control Together

In [ ]:
final_query = f"""
SELECT DISTINCT
    person_id, cr1_first_age first_age, 1 hypothyroidism
FROM ({final_case_query})
UNION ALL
SELECT DISTINCT
    person_id, first_TSH_age first_age, 0 hypothyroidism
FROM ({final_ctrl_query})
"""

In [ ]:
count_query(final_query)

In [ ]:
count_grp_query(final_query, 'hypothyroidism')

In [ ]:
final_case_ctrl_pd = pd.read_gbq(final_query, dialect="standard", progress_bar_type="tqdm_notebook")

In [ ]:
final_case_ctrl_pd['hypothyroidism'].value_counts()

In [ ]:
fs = gcsfs.GCSFileSystem(requester_pays=True)
#with fs.open('gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv') as f:
with fs.open('gs://vwb-aou-datasets-controlled/v7/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv') as f:
    ancestry = pd.read_csv(f, sep='\t')[['research_id', 'ancestry_pred']]

final_query_df = pd.read_gbq(final_query, dialect="standard")

final_case_ctrl_pd_wgs = final_query_df.merge(
    ancestry,
    how='inner',
    left_on='person_id',
    right_on='research_id'
)[['person_id', 'hypothyroidism', 'ancestry_pred']]

In [ ]:
final_case_ctrl_pd_wgs.hypothyroidism.value_counts().to_frame()

In [ ]:
fs = gcsfs.GCSFileSystem(requester_pays=True)
with fs.open(f'{bucket}/hypothyroid_data/huan_phenotype_v4.csv', 'w') as f:
    final_case_ctrl_pd_wgs.to_csv(f, index=False)

In [ ]:
final_case_ctrl_pd_wgs.groupby(by=['hypothyroidism', 'ancestry_pred']).count()

In [ ]:
current1 = datetime.now()
used_time1 = str(current1 - start)
used_time1

# Covars


In [ ]:
fin = final_case_ctrl_pd_wgs.merge(
    pd.read_gbq(f'''
        SELECT DISTINCT
            person_id,
            sex_at_birth,
            age_at_cdr
        FROM
            {version}.cb_search_person
    '''),
    how = 'left',
    on = 'person_id'
)

In [ ]:
fin = pd.get_dummies(fin, columns=['sex_at_birth']).rename(columns={'hypothyroidism': 'Hypothyroidism'})[['person_id', 'Hypothyroidism', 'ancestry_pred', 'age_at_cdr', 'sex_at_birth_Female']]

In [ ]:
fs = gcsfs.GCSFileSystem(requester_pays=True)
with fs.open(f'{bucket}/hypothyroid_data/huan_phenotype_v4_covars.csv', 'w') as f:
    fin.to_csv(f, index=False)